# Commande utiles shell

In [ ]:
# compter des lignes

wc -l test.txt

# chercher un mot

grep "rs123" fichier.txt

# Décompresser a la volée 

zcat chr11.vcf.gz | head



# Type fichier

## VCF

un ficher **VCF** contient des variants génétiques, souvent : SNP, indels, génotypes pour plusieurs individus

exemple : 

```

##fileformat=VCFv4.2
##source=bcftools
#CHROM  POS     ID       REF ALT QUAL FILTER INFO FORMAT sample1 sample2
1       10583   rs1      G   A   29   PASS   .    GT     0/1     0/0
1       10611   rs2      C   G   45   PASS   .    GT     1/1     0/1

```

Les lignes commencent par # sont le header, 2 types :

- Métadonnées ##
- Lignes principales des colonnes #

### Les colonnes a connaitre : 

CHROM -> Chromosome

POS -> Position sur le chromosome

ID -> Nom du variant, souvent un rsID

REF -> Allèle de référence

ALT -> Allèle alternatif

QUAL -> Qualité du variant

FILTER -> Statut du filtre
    exemple : PASS = OK  |  autre chose = variant suspect ou filtré

INFO -> Infos complémentaires

FORMAT -> Indique ce que contiennent les colonnes échantillons


### Comprendre les génotypes :

0/0 -> homozygote référence

0/1 -> hétérozygote

1/1 -> homozygote alternatif

./. -> génotype manquant

### Ce qu’on filtre souvent dans un VCF

En pratique, on retire souvent :

- les variants trop rares selon l’objectif
- les variants avec trop de données manquantes
- les variants de mauvaise qualité
- les variants multialléliques selon le pipeline
- les indels si on veut un GWAS SNP simple

# Concept Génétique

## MAF

Fréquence de l'allèle minoritaire 

Par exemple : A = 95% G = 5%
Alors : MAF = 0.05

## Missingness

C'est le taux de données manquantes

Il y a 2 côtés : 
- le missingness par variant -> SNP mal génotypé dans bcp d'individus 
- le missingess par individu -> un individu avec bcp de SNPs manquants 

En pratique on filtre souvent : 
- variants : geno 0.05
- individus : mind 0.05

## HWE 

**Hardy-Weinberg Equilibrium**

La loi de Hardy-Weinberg stipule que les fréquences des allèles et des génotypes restent constantes dans une population idéale. 

Cette loi s’applique seulement si une population est grande, qu’il n’y a ni mutation, ni migration, ni sélection naturelle ou sexuelle. 

Les deux équations fondamentales sont p + q = 1 (pour les fréquences alléliques) et p² + 2pq + q² = 1 (pour les fréquences génotypiques).
En connaissant la fréquence d’un phénotype récessif (q²), il est possible de calculer les fréquences des autres allèles et génotypes. 
Le principe sert de référence pour les scientifiques afin de déterminer si une population est en cours d’évolution.

En savoir plus sur: https://jeretiens.net/le-principe-dequilibre-de-hardy-weinberg/


test statistique pour détecter des variants bizarres  avec : HWE p < 1e-6



## LD 

**Linkage Disequilibrium**  -> Situation dans laquelle deux gènes sont trouvés ensemble dans une population à une fréquence supérieure à celle prédite par le produit de leur fréquence individuelle

2 variant proche peuvent être corrélés et dans certains cas on veut évité d'avoir trop de SNPs redondants, donc on fait du LD pruning 

## Le génotype (GT)

Le génotype indique **quels allèles possède l'individu** a une postion 

```
CHROM POS      REF ALT GT
22    16050075 A   G   0|0
```

Signification des chiffres :

0 = REF

1 = ALT

2 = ALT2

3 = ALT3

Donc :

0|0 → A / A

0|1 → A / G

1|1 → G / G

Cas multi-allélique : REF = A ALT = G,T **alors** 0=A 1=G 2=T **donc** 1|2 = G/T 0|2 = A/T 2|2 = T/T


# bcftools 

## Lecture VCF

**Attention important** 

- view = filtrer / sélectionner
- query = extraire / formater
- head = voir seulement quelques lignes

In [ ]:
# voir header (-h --header-only)

bcftools view -h files.vcf.gz 
bcftools view -h files.vcf.g | less # car souvent trop long



# voir uniquement les variants (-H --no-header)

bcftools view -H files.vcf.gz 
bcftools view -H files.vcf.gz | head -5 # si que les 5 premiers 

# lister les samples (-l list-samples)

bcftools query -l files.vcf.gz 

# extraire colonnes 

bcftools query -f 'FORMAT' file.vcf.gz 
bcftools query -f '%CHROM\t%POS\t%REF\t%ALT\n' files.vcf.gz

# compter les samples 

bcftools query -l files.vcf.gz | wc -l 

# compter les variants

bcftools view -H files.vcf.gz | wc -l 

# voir les chromosomes présents 

bcftools query -f '%CHROM\n' files.vcf.gz | sort | uniq 

## Filtrer un VCF   

**Attention**

- Avec view : header par défaut *donc* -H pour l’enlever
- Avec query : jamais de header *donc* -H inutile

**La commande -v**

- Cette commande accepte : snps, indels (ins et del), mnps, other
- Elle peut accepter plusieurs : -v TYPE1,TYPE2,TYPE3
- Si on veux exclure, à la place de filter : -V TYPE

*A savoir :* 

*- mnp (multi-nucleotide polymorphism) signifie que plusieurs bases changent mais même longueur*

*- other : variants complexes, multialléliques complexes, structural variants courts, combinaisons SNP + indel*

**La commande -o**

- sert à écrire dans un fichier sans bcftools écrit dans le terminal 

**La commande -O**

- sert à choisir le format de sortie 
- Ov  → VCF (texte)
- Oz  → VCF compressé (.vcf.gz)
- Ob  → BCF (binaire)
- Ou  → BCF non compressé

**Indexation**

Il existe 2 types d’index courants pour VCF.gz :

- .tbi -> ancien format, très courant
- .csi -> plus général, plus robuste pour de grons contigs/grandes coordonnées 

```bcftools index``` peut produire un .csi par défaut selon le contexte et la version.

**La commande -s**

Permet de filtrer les samples :
- -s → samples directement sur la ligne de commande
- -S → samples depuis un fichier texte




In [ ]:
# garder SNP uniquement 

bcftools view -v snps files.vcf.gz 

# garder SNP uniquement sans Headers 

bcftools view -H -v snps files.vcf.gz 

# Filtre SNP/extraire certaine colonnes/limite 3 lignes

bcftools view -v snps chr22.vcf.gz | bcftools query -f '%CHROM\t%POS\t%REF\t%ALT\n' | head -3 

# Compter les SNPs

bcftools view -H -v snps files.vcf.gz | wc -l

# Afficher les 3 premiers SNPs bialléliques 

bcftools view -m2 -M2 -v snps  chr22.vcf.gz | bcftools query -f '%CHROM\t%POS\t%REF\t%ALT\n' | head -3
    # -m2 : min 2 allèle (REF + 1ALT)
    # -M2 : max 2 allèle (donc 1 seul ALT)
    # donc on bialléliques uniquement 

# Compter le nombre allèle biallélique 

bcftools view -H -m2 -M2 -v snps  chr22.vcf.gz | wc -l

# Extraire une région génomique # -r CHR:START-END

bcftools view -r 22:16050000-16060000 chr22.vcf.gz | bcftools query -f '%CHROM\t%POS\t%REF\t%ALT\n' | head -3
    # avant d'utiliser -r, toujours vérifier le nom du chromosome : 22,chr22,Chr22,NC_000001.22
    # bcftools query -f '%CHROM\n' chr22.vcf.gz | head
    # ou bcftools view -h chr22.vcf.gz | grep contig

# Compter des variants dans une région

bcftools view -H -r 22:16050000-16060000 chr22.vcf.gz | wc -l

# Sauvegarder une région dans un VCF 

bcftools view -r 22:16050000-16060000 chr22.vcf.gz -Oz -o chr22_portion.vcf.gz 

# Indexer un VCF 

bcftools index chr22_portion.vcf.gz 
    # Indexé : filtrage rapide, accès par région, utilisation avec plink/bcftools 

# Filtre a partir des samples 

bcftools view -s SAMPLE file.vcf.gz
bcftools view -s SAMPLE1,SAMPLE2,SAMPLE3 file.vcf.gz
bcftools view -S liste_samples.txt file.vcf.gz

    # exemple d'utlisation 
    ## Afficher sur 3 variant le génotype pour un ou plusieurs samples 
    bcftools view -s HG00096 chr22.vcf.gz | bcftools query -f '%CHROM\t%POS\t%REF\t%ALT\t[%GT]\n'| head -3
    bcftools view -s HG00096,HG00097,HG00099 chr22.vcf.gz | bcftools query -f '%CHROM\t%POS\t%REF\t%ALT\t[%GT\t]\n'| head -3 
                ### ne pas oublié après le GS \t ou mettre un espace pour espacer les génotype entre patient 